<a href="https://colab.research.google.com/github/ijazkhan0351-bot/Ijazweek1-ml-assignment/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ijazkhan0351-bot/Ijazweek1-ml-assignment/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip install -q duckdb huggingface_hub

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"

schema_df = con.sql(f"""
    DESCRIBE SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet') LIMIT 1
""").df()
print("Connected. Columns available:")
print(schema_df)

Connected. Columns available:
                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES 

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item's performance on one report date, for one client
(the grain of `fact_content_daily_performance` is client_hash_id ×
content_hash_id × report_date). Time window: month=2026-03 — a mid-panel
month, not the sealed final month (June 2026), so I avoid building label
logic on the natural outcome window.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify: grain check (no duplicate rows per client+content+date)
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as row_count
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Rows where grain breaks (should be empty):")
print(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows where grain breaks (should be empty):
Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, row_count]
Index: []


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features (knowable at decision time):** gsc_impressions, gsc_clicks,
gsc_avg_position — all built only from data up to and including the
decision month.

**Label / proxy:** whether performance is declining, measured by comparing
this month to a later month — usable only for evaluation, never as an
input feature, since it isn't known when a refresh decision is made.

**Context (not used as predictive features):** client_hash_id,
content_hash_id — identifiers, kept for joining/grouping only, excluded
from the model itself to avoid overfitting to specific pages.

**Excluded:** any future-month metric, and any AI-referral columns
(ai_chatgpt, ai_perplexity, etc.) for this lane, since my claim is specifically
about search/refresh signals, not AI-referral traffic — mixing them in
would blur what I'm actually testing.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Build a small feature frame for this month
features_df = con.sql(f"""
    SELECT
        content_hash_id,
        AVG(gsc_impressions) as avg_impressions,
        AVG(gsc_clicks) as avg_clicks,
        AVG(gsc_avg_position) as avg_position,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) as ctr
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
    LIMIT 1000
""").df()

# STEP 1: add ONE label-derived column on purpose (the trap)
features_df["leaky_col"] = features_df["avg_clicks"] * 1.15  # simulates future-derived info
print("With the leaky column included, a quick score would look unrealistically strong.")

# STEP 2: delete it, keep only the honest features
features_df = features_df.drop(columns=["leaky_col"])
print("Leaky column removed — keeping only features knowable at decision time.")
features_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

With the leaky column included, a quick score would look unrealistically strong.
Leaky column removed — keeping only features knowable at decision time.


,content_hash_id,avg_impressions,avg_clicks,avg_position,ctr
0,content_7a105f548d9c6916,210.419355,0.225806,7.209549,0.001073
1,content_a3ea9792f793ec72,14.612903,0.000000,2.987198,0.000000
2,content_36c36abc7650d7af,181.612903,0.193548,6.724039,0.001066
3,content_a7da352b73b02668,159.483871,0.419355,7.244844,0.002629
4,content_1855a661b4d36130,13.838710,0.032258,4.209227,0.002331


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Two more claims verified directly against the warehouse: how many rows
this slice actually has and what date span it covers, and how many rows
survive an availability filter using `IS TRUE`.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Row count + date span
span_check = con.sql(f"""
    SELECT
        COUNT(*) as total_rows,
        MIN(report_date) as earliest_date,
        MAX(report_date) as latest_date,
        COUNT(DISTINCT content_hash_id) as unique_content_items
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(span_check)

# Availability check with IS TRUE
avail_check = con.sql(f"""
    SELECT COUNT(*) as available_rows
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
""").df()
print(avail_check)

   total_rows earliest_date latest_date  unique_content_items
0     9841378    2026-03-01  2026-03-31                331437
   available_rows
0         3611061


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data can't tell me everything. It's an unbalanced panel — per-client
history depth differs, and `client_has_gsc`/`client_has_ga4` flags show
some clients only have partial data source coverage, so a page missing
GSC data isn't necessarily "no traffic" — it may just mean that client
never connected Search Console. This could skew any pattern I find toward
clients with fuller data coverage. This data is observational only — it
shows correlation between signals and decline, never causation.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Supporting check: how many rows have partial data availability?
missing_check = con.sql(f"""
    SELECT
        COUNT(*) as total_rows,
        SUM(CASE WHEN gsc_data_available THEN 1 ELSE 0 END) as gsc_available_rows,
        SUM(CASE WHEN ga4_data_available THEN 1 ELSE 0 END) as ga4_available_rows
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(missing_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  gsc_available_rows  ga4_available_rows
0     9841378           3611061.0            413966.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.